In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
import os
import torch
import torchaudio
import parselmouth

In [2]:
files = pd.read_csv('nonAfrica.csv', header=0, index_col=0)

In [3]:
audios = {file:parselmouth.Sound('.././data/audios_development/'+file) for file in files['path']}

In [4]:
def computeF0(audio):
    pitch = audio.to_pitch()
    return pd.Series([pitch.selected_array['frequency'].mean(), 
            pitch.selected_array['frequency'].std(), 
            pitch.selected_array['frequency'].max(), 
            pitch.selected_array['frequency'][pitch.selected_array['frequency']>0].min()],
                     index=['meanF0', 'stdF0', 'maxF0', 'minF0'])

In [5]:
def computeSpectralCharacteristic(audio):
    spectrum = audio.to_spectrum() 
    # spectral_rolloff
    
    return pd.Series([spectrum.get_center_of_gravity(), spectrum.get_standard_deviation(), spectrum.get_skewness()],
                     index=['center_of_gravity', 'standard_deviation', 'skewness'])

In [6]:
def signalCharacteristics(audio):
    return pd.Series([audio.get_intensity(), audio.get_energy(), audio.get_energy_in_air(), audio.get_power(), audio.get_power_in_air()], 
                     index=['intensity', 'energy', 'energy_in_air', 'power', 'power_in_air'])

In [7]:
def computeRMSDuration(audio):
    return pd.Series([audio.get_root_mean_square(), audio.values.shape[1]/22050], index=['rms', 'duration'])

In [8]:
def getMFCC(audio):
    mfcc = audio.to_mfcc().extract_features().values[:-1, :]
    return pd.Series(np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1), mfcc.max(axis=1)], axis=0),
                     index=[f'mfcc_mean_{i}' for i in range(0, 3)]+[f'mfcc_std_{i}' for i in range(0, 3)]+[f'mfcc_max_{i}' for i in range(0, 3)])

In [9]:
def computeJitterShimmerHNR(key, files):
    return pd.Series(files[files['path']==key][['jitter', 'shimmer', 'hnr']].values[0], 
                     index=['jitter', 'shimmer', 'hnr'])

In [10]:
def computeZCR(audio):
    zcr = np.array([audio.get_nearest_zero_crossing(i) for i in range(0, audio.values.shape[1]//22050)])
    return pd.Series([zcr.mean(), zcr.std()], index=['zcrMean', 'zcrStd'])

In [26]:
from scipy.signal import stft

def compute_chroma(audio, sample_rate=22050, n_fft=2048, hop_length=512):
    # Calcola la STFT dello spettro
    _, _, Zxx = stft(audio, fs=sample_rate, nperseg=n_fft, noverlap=n_fft-hop_length)
    spectrogram = np.abs(Zxx)
    
    # Definisci il mappaggio delle frequenze alle note
    chroma_filter = np.zeros((12, n_fft // 2 + 1))
    frequencies = np.linspace(0, sample_rate // 2, n_fft // 2 + 1)
    A440 = 440.0  # Frequenza di riferimento
    
    for i, freq in enumerate(frequencies):
        if freq > 0:
            # Calcola il numero di semitoni rispetto al LA centrale (A440)
            note = int(np.round(12 * np.log2(freq / A440))) % 12
            chroma_filter[note, i] += 1
    
    # Applica il filtro alle frequenze e somma le energie per ciascuna banda
    chroma = np.dot(chroma_filter, spectrogram)
    chroma = chroma / (np.sum(chroma, axis=0, keepdims=True) +1e-12)
    return pd.Series(np.concatenate((chroma.mean(axis=1), chroma.std(axis=1), chroma.max(axis=1)), axis=0), 
            index=[f'chroma_mean_{i}' for i in range(12)] + [f'chroma_std_{i}' for i in range(12)] + [f'chroma_max_{i}' for i in range(12)])

In [12]:
def computeSilenceDuration(audio):
    dur = audio.total_duration
    audio = audio.values
    return pd.Series(audio[audio < audio.mean()-3*audio.std()].shape[0]/audio.shape[1]*dur, index=['silenceDuration'])

In [27]:
data = pd.concat([pd.DataFrame({file: pd.concat([
                                computeF0(audios[file]), 
                                computeSpectralCharacteristic(audios[file]),
                                # signalCharacteristics(audios[file]), 
                                # computeRMSDuration(audios[file]),
                                computeSilenceDuration(audios[file]),
                                getMFCC(audios[file]), 
                                computeJitterShimmerHNR(file, files),
                                computeZCR(audios[file]), 
                                compute_chroma(audios[file].values[0], 22050)]
                                  ) for file in audios}).T,
                        files.set_index(files['path'])['age']], axis=1)

In [28]:
# data = pd.read_csv('nonAfricaFeatures.csv', header=0, index_col=0)

In [33]:
data

,meanF0,stdF0,maxF0,minF0,center_of_gravity,standard_deviation,skewness,silenceDuration,mfcc_mean_0,mfcc_mean_1,...,chroma_max_3,chroma_max_4,chroma_max_5,chroma_max_6,chroma_max_7,chroma_max_8,chroma_max_9,chroma_max_10,chroma_max_11,age
1.wav,105.337970,121.841823,587.415757,78.107618,1125.768753,1213.670110,4.690098,0.492653,5943.947241,63.619388,...,0.568364,0.359980,0.495048,0.344745,0.372825,0.265705,0.273250,0.309603,0.252379,24.0
2.wav,107.219543,101.956515,416.290578,77.474473,476.688561,509.511523,5.272371,0.196100,8342.378031,67.863597,...,0.419268,0.317141,0.210184,0.173805,0.235624,0.333791,0.279595,0.426395,0.235092,22.5
3.wav,116.533228,119.920416,587.238408,93.960406,647.506469,1330.941691,5.533402,0.157324,6772.904887,69.078752,...,0.408443,0.364292,0.295528,0.168455,0.269057,0.674836,0.214311,0.312233,0.270297,22.0
4.wav,130.331735,109.496279,469.625117,85.656547,1013.801094,1862.988023,3.374334,0.126735,10316.574580,66.876994,...,0.422254,0.342949,0.414793,0.375782,0.332308,0.383218,0.417511,0.331221,0.246299,22.0
5.wav,60.492711,78.679495,597.502669,77.429089,608.678429,1098.335148,4.576950,0.287868,10881.592207,73.878072,...,0.391739,0.212762,0.320254,0.163460,0.223516,0.546847,0.388673,0.242281,0.218660,22.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2924.wav,90.350657,89.629836,578.264844,76.401820,534.117556,987.835844,7.606841,0.304422,8459.032375,69.423915,...,0.442970,0.265641,0.330319,0.212856,0.213289,0.485945,0.219229,0.382038,0.220884,27.0
2926.wav,102.840631,107.826812,550.534575,81.633244,610.067203,674.893843,5.910978,0.320437,1339.492388,47.177903,...,0.537602,0.409030,0.529391,0.533836,0.489730,0.279981,0.506623,0.483061,0.492954,22.0
2928.wav,100.281554,91.416195,562.003333,80.163402,653.714379,1219.474166,3.905207,0.542562,5456.890270,61.601533,...,0.248012,0.162988,0.328727,0.135851,0.295153,0.370609,0.262644,0.330186,0.249162,39.0
2929.wav,60.221861,66.950635,593.541250,75.313048,785.066539,1267.951417,3.155046,0.157959,12082.535236,70.757569,...,0.379535,0.229924,0.343581,0.192973,0.225624,0.577275,0.168529,0.241620,0.210686,24.0


In [34]:
temp = data.copy()
age = temp['age']
temp = temp.drop(columns=['age'])


In [35]:
temp = pd.DataFrame(StandardScaler().fit_transform(temp), columns=temp.columns, index=temp.index)

In [39]:
temp.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1695 entries, 1.wav to 2932.wav
Data columns (total 58 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   meanF0              1695 non-null   float64
 1   stdF0               1695 non-null   float64
 2   maxF0               1695 non-null   float64
 3   minF0               1695 non-null   float64
 4   center_of_gravity   1695 non-null   float64
 5   standard_deviation  1695 non-null   float64
 6   skewness            1695 non-null   float64
 7   silenceDuration     1695 non-null   float64
 8   mfcc_mean_0         1695 non-null   float64
 9   mfcc_mean_1         1695 non-null   float64
 10  mfcc_mean_2         1695 non-null   float64
 11  mfcc_std_0          1695 non-null   float64
 12  mfcc_std_1          1695 non-null   float64
 13  mfcc_std_2          1695 non-null   float64
 14  mfcc_max_0          1695 non-null   float64
 15  mfcc_max_1          1695 non-null   float64
 16  mfc

In [37]:
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA 
pipeline = make_pipeline(StandardScaler(), 
                         PCA(n_components=40),
                         RandomForestRegressor(n_jobs=-1))

kf = KFold(n_splits=10, shuffle=True, random_state=42)

cross_val_score(pipeline, temp, age, cv=kf, scoring='neg_mean_absolute_error').mean()

ValueError: 
All the 10 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\utente\OneDrive\Desktop\Magistrale\01TWZSM_DataScienceLab\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\utente\OneDrive\Desktop\Magistrale\01TWZSM_DataScienceLab\.venv\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\utente\OneDrive\Desktop\Magistrale\01TWZSM_DataScienceLab\.venv\Lib\site-packages\sklearn\pipeline.py", line 469, in fit
    Xt = self._fit(X, y, routed_params)
         ^^^

In [ ]:
clf = RandomForestRegressor(n_jobs=-1)
clf.fit(temp, data['age'])

for name, importance in sorted(zip(temp.columns, clf.feature_importances_), key=lambda x: x[1], reverse=True):
    print(f'{name}: {importance}')

zcrStd: 0.05280628640631185
zcrMean: 0.041750427124830576
silenceDuration: 0.037412870106319364
chroma_mean_9: 0.032824356762113746
mfcc_std_1: 0.02884744772862839
chroma_mean_5: 0.028621627553445342
minF0: 0.025355088989958613
mfcc_mean_1: 0.024301274399208688
jitter: 0.024056688577305967
chroma_mean_7: 0.024018563381955262
chroma_max_10: 0.020940531399127747
chroma_max_11: 0.02092524284238378
shimmer: 0.02024384685435543
chroma_std_4: 0.01933423479680965
chroma_std_5: 0.019089577114857606
chroma_max_3: 0.017997963682805672
chroma_max_1: 0.017793634002172595
mfcc_max_2: 0.017401601702770088
maxF0: 0.017057212139771454
chroma_max_2: 0.017012995125243995
chroma_mean_3: 0.016341046609289072
hnr: 0.016301487002438875
chroma_std_8: 0.016108372663062698
chroma_max_9: 0.0160988478318054
chroma_mean_0: 0.015928223386475125
mfcc_max_1: 0.015656523653694413
chroma_mean_4: 0.014923980934272822
chroma_max_8: 0.014465578467808257
chroma_mean_10: 0.014343065960640295
meanF0: 0.013768786840199212
ch

In [ ]:
temptemp = pd.concat([temp.drop(columns=['chroma_std_0', 'chroma_std_1', 'chroma_std_2',
       'chroma_std_3', 'chroma_std_4', 'chroma_std_5', 'chroma_std_6',
       'chroma_std_7', 'chroma_std_8', 'chroma_std_9', 'chroma_std_10',
       'chroma_std_11',
       'chroma_max_0',  'chroma_max_1', 'chroma_max_2',
       'chroma_max_3', 'chroma_max_4', 'chroma_std_5', 'chroma_max_6',
       'chroma_max_7', 'chroma_max_8', 'chroma_std_9', 'chroma_max_10',
       'chroma_max_11',
       'mfcc_max_0', 'mfcc_max_1', 'mfcc_max_2']), 
                      pd.Series(temp[['chroma_std_0', 'chroma_std_1', 'chroma_std_2',
       'chroma_std_3', 'chroma_std_4', 'chroma_std_5', 'chroma_std_6',
       'chroma_std_7', 'chroma_std_8', 'chroma_std_9', 'chroma_std_10',
       'chroma_std_11']].mean(axis=1), name='chromaStd'),
                      pd.Series(temp[['chroma_max_0',  'chroma_max_1', 'chroma_max_2',
       'chroma_max_3', 'chroma_max_4', 'chroma_std_5', 'chroma_max_6',
       'chroma_max_7', 'chroma_max_8', 'chroma_std_9', 'chroma_max_10',
       'chroma_max_11']].mean(axis=1), name='chromaMax'),
                      pd.Series(temp[['mfcc_max_0', 'mfcc_max_1', 'mfcc_max_2']].mean(axis=1), name='mfccMax')], axis=1)
temptemp

,meanF0,stdF0,maxF0,minF0,center_of_gravity,standard_deviation,skewness,silenceDuration,mfcc_mean_0,mfcc_mean_1,mfcc_mean_2,mfcc_std_0,mfcc_std_1,mfcc_std_2,jitter,shimmer,hnr,zcrMean,zcrStd,chroma_mean_0,chroma_mean_1,chroma_mean_2,chroma_mean_3,chroma_mean_4,chroma_mean_5,chroma_mean_6,chroma_mean_7,chroma_mean_8,chroma_mean_9,chroma_mean_10,chroma_mean_11,chroma_max_5,chroma_max_9,chromaStd,chromaMax,mfccMax
1.wav,0.820895,1.131923,0.573251,-0.311759,0.270654,-0.131407,-0.142846,0.902404,-0.620444,-1.056695,1.752186,-0.496825,0.351458,0.544964,-1.490842,-1.185036,-1.268093,NaN,NaN,-1.117323,0.578929,1.755139,-0.070952,1.871776,0.620463,1.445166,0.619275,-1.221559,0.185890,-0.712063,0.046603,2.021680,0.277857,0.435930,0.804674,0.393278
2.wav,0.886474,0.338809,-1.068385,-0.360200,-0.654921,-0.988488,0.059064,-0.901106,0.198471,-0.303317,-1.170429,0.156734,1.014377,-1.388188,0.330115,-0.691115,0.653242,NaN,NaN,1.520956,-0.021226,-0.810729,-0.948720,-0.885990,-1.172005,-0.572390,-1.103863,0.053599,-0.599866,2.863505,0.097690,-1.772689,0.367460,0.101751,0.041322,-0.580665
3.wav,1.211085,1.055289,0.571550,0.901121,-0.411338,0.011332,0.149580,-1.136922,-0.337407,-0.087617,-0.794150,-0.628878,0.316285,0.112530,-0.659810,0.157123,0.056068,NaN,NaN,0.239219,1.128586,-0.023238,1.439639,-0.778144,-2.207155,-0.583379,-1.338689,0.640948,-1.039968,-0.105406,-0.217783,-0.635901,-0.554389,0.266203,0.102061,-0.497320
4.wav,1.692006,0.639528,-0.556737,0.265801,0.110990,0.658923,-0.599104,-1.322955,0.872536,-0.478446,1.502472,0.886481,0.562380,1.803299,-0.985064,-0.466508,2.232380,NaN,NaN,0.461398,0.801793,-0.367703,-1.217665,0.527643,0.215702,1.379246,0.258667,-0.827683,0.358351,0.338031,0.651446,0.952700,2.314900,0.873603,0.631780,1.273725
5.wav,-0.742102,-0.589582,0.670017,-0.363673,-0.466706,-0.271789,-0.182082,-0.343008,1.065454,0.764298,-0.290625,1.151284,0.596013,0.656833,0.752320,0.353536,0.994234,NaN,NaN,0.993074,-0.943459,-1.299335,-0.194494,-1.100739,-0.771522,-0.563363,-0.168639,0.925341,0.129821,0.292872,0.203687,-0.306556,1.907695,-0.288833,-0.471509,0.824813
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2924.wav,0.298540,-0.152834,0.485465,-0.442268,-0.573029,-0.406286,0.868571,-0.242338,0.238301,-0.026349,-1.561610,-0.088143,0.260505,-1.006650,1.023923,0.229402,0.535541,NaN,NaN,-0.869903,-1.844226,-1.172627,1.176845,-1.180681,1.764922,-1.834894,-0.527819,1.896794,-0.663848,0.141874,-2.206783,-0.172495,-0.484941,0.021234,0.028145,-0.416099
2926.wav,0.733855,0.572942,0.219443,-0.042017,-0.464726,-0.787189,0.280509,-0.144937,-2.192578,-3.975182,-0.194446,-2.227716,-2.757752,-0.831111,-1.259869,-0.996589,-0.471648,-1.217458,-1.217822,-3.480189,-1.504829,1.234768,-0.043482,2.585351,1.885592,3.927123,1.691194,-1.650909,1.138267,0.159502,1.045925,2.479134,3.573197,1.578920,1.664891,-1.857788
2928.wav,0.644663,-0.081586,0.329465,-0.154473,-0.402486,-0.124343,-0.415017,1.205931,-0.786744,-1.414880,-0.600630,-0.376107,-1.415636,-0.511541,0.598356,0.425165,1.645979,NaN,NaN,-0.481372,-0.936964,-0.573432,-1.227796,-1.189686,0.271252,-0.970516,1.697544,0.481020,1.237045,1.377674,0.070487,-0.193692,0.128101,-0.196686,-0.296150,-0.508757
2929.wav,-0.751542,-1.057381,0.632015,-0.525569,-0.215181,-0.065338,-0.675145,-1.133061,1.475502,0.210385,-0.174879,1.344662,0.740787,-0.096069,0.062006,0.007483,-0.071968,NaN,NaN,0.056171,-0.476930,-0.045867,0.465378,-0.104173,0.156943,0.478967,0.211100,0.307102,-0.647427,-0.741697,-0.400576,0.004159,-1.200853,-0.533147,-0.595155,0.062971


In [ ]:
print(cross_val_score(RandomForestRegressor(n_jobs=-1), temptemp, 
                age, cv=10, scoring='neg_mean_absolute_error').mean())


for name, imp in sorted(zip(temptemp.columns, RandomForestRegressor(n_jobs=-1).fit(temptemp, age).feature_importances_), key=lambda x: x[1], reverse=True):
    print(f'{name}: {imp}')

-10.386645701357466
silenceDuration: 0.1016446150050201
chroma_mean_5: 0.06083756112584231
minF0: 0.05316411289294977
mfcc_std_1: 0.050586689501387526
mfcc_mean_1: 0.04615232425682087
chroma_mean_9: 0.03331680840652916
chroma_max_9: 0.03240356759362807
chroma_mean_3: 0.028907635470625883
hnr: 0.027639895408929086
chroma_mean_11: 0.02740701253001651
chroma_mean_4: 0.027406216811850643
shimmer: 0.027292910321297825
chroma_mean_7: 0.02643516967961849
jitter: 0.026420691635438538
chroma_mean_10: 0.026034814199931578
mfcc_mean_0: 0.02580719637577123
mfccMax: 0.024733064746728324
chroma_mean_0: 0.024548593807992607
chroma_mean_1: 0.024394278925667107
chroma_max_5: 0.023768927979044577
chroma_mean_2: 0.023714589678247296
chroma_mean_6: 0.023045883852698876
chroma_mean_8: 0.02276444287240745
maxF0: 0.02037628166362395
mfcc_std_2: 0.020313112316636527
standard_deviation: 0.019845369777072203
skewness: 0.019775938935696877
stdF0: 0.01915169415146857
mfcc_mean_2: 0.018560518528469516
mfcc_std_0: 

In [ ]:
cross_val_score(RandomForestRegressor(n_jobs=-1), files.drop(columns=['path', 'age', 'ethnicity']), files['age'], cv=10, scoring='neg_mean_absolute_error').mean()

np.float64(-10.838645144448313)

In [292]:
data

,meanF0,stdF0,maxF0,minF0,center_of_gravity,standard_deviation,skewness,silenceDuration,mfcc_mean_0,mfcc_mean_1,mfcc_mean_2,mfcc_std_0,mfcc_std_1,mfcc_std_2,mfcc_max_0,mfcc_max_1,mfcc_max_2,jitter,shimmer,hnr,zcrMean,zcrStd,chroma_mean_0,chroma_mean_1,chroma_mean_2,chroma_mean_3,chroma_mean_4,chroma_mean_5,chroma_mean_6,chroma_mean_7,chroma_mean_8,chroma_mean_9,chroma_mean_10,chroma_mean_11,chroma_std_0,chroma_std_1,chroma_std_2,chroma_std_3,chroma_std_4,chroma_std_5,chroma_std_6,chroma_std_7,chroma_std_8,chroma_std_9,chroma_std_10,chroma_std_11,chroma_max_0,chroma_max_1,chroma_max_2,chroma_max_3,chroma_max_4,chroma_max_5,chroma_max_6,chroma_max_7,chroma_max_8,chroma_max_9,chroma_max_10,chroma_max_11,age
1.wav,75.802961,117.835151,578.538578,130.497292,1649.296779,1672.027330,3.091895,0.425442,27308.670357,90.903457,16.010235,66742.513255,58.962539,10.528454,9.246653e+05,515.558076,30.999992,0.013795,0.082725,-123.999726,16.999842,10.099506,0.081007,0.064205,0.069960,0.092690,0.070478,0.087835,0.070136,0.107249,0.173912,0.062820,0.058810,0.060899,0.091241,0.061127,0.068160,0.085851,0.068353,0.081417,0.083589,0.101443,0.255539,0.072005,0.057385,0.057403,0.682357,0.456009,0.719288,0.668185,0.473556,0.558414,0.813626,0.687171,0.943517,0.650257,0.411964,0.340413,24.0
2.wav,72.628349,96.539640,277.558361,78.882468,958.178143,748.272676,2.382307,0.270703,23892.126348,97.463155,9.647331,56774.265995,56.191397,8.369566,1.147554e+06,476.424295,30.998979,0.025349,0.096242,-86.928478,11.000024,6.633266,0.128232,0.070557,0.061471,0.094622,0.062454,0.081837,0.059010,0.070759,0.096662,0.072724,0.097632,0.104042,0.093264,0.056509,0.057880,0.073485,0.057554,0.053085,0.059527,0.059018,0.115813,0.066283,0.084847,0.089683,0.675690,0.417362,0.416860,0.556068,0.412294,0.394035,0.519111,0.531834,0.911449,0.399933,0.668522,0.497765,22.5
3.wav,83.555433,113.581832,586.393148,93.904151,1511.520803,2153.010836,1.915487,0.203356,23395.320693,95.883027,11.085801,60354.163135,53.617316,9.328402,1.062133e+06,533.599839,30.998794,0.019067,0.119456,-98.450670,10.000070,6.055004,0.127460,0.078105,0.069986,0.108449,0.048093,0.069491,0.051003,0.074467,0.143688,0.071525,0.082477,0.075257,0.107952,0.075368,0.068570,0.088131,0.051995,0.053989,0.049838,0.075380,0.184099,0.065071,0.072850,0.073301,0.845364,0.455359,0.418754,0.634017,0.327528,0.395065,0.323977,0.846861,0.903967,0.557796,0.460150,0.401491,22.0
4.wav,99.814653,115.958678,592.696608,83.787581,2758.567510,3063.538710,1.052278,0.235782,23093.979368,92.106786,13.511746,56823.548023,49.670962,9.916377,8.730858e+05,411.860362,30.999990,0.017004,0.102389,-56.459762,10.500104,6.344293,0.112082,0.063974,0.065957,0.075151,0.073172,0.103859,0.074832,0.083258,0.121329,0.070561,0.069193,0.086631,0.112803,0.065854,0.062180,0.067638,0.066665,0.078302,0.071578,0.071353,0.194133,0.062817,0.054027,0.082971,0.697082,0.482121,0.396752,0.476014,0.420051,0.507776,0.458929,0.637906,0.933931,0.437091,0.323343,0.546532,22.0
5.wav,44.703150,87.716014,599.179923,77.452722,1475.223518,1569.424306,1.505394,0.228889,30330.557663,106.085947,10.672633,67816.876750,49.375923,8.630707,1.221176e+06,407.322076,30.999871,0.028027,0.124831,-80.349204,9.000207,5.477031,0.118305,0.082151,0.060632,0.088217,0.059804,0.089203,0.071056,0.078661,0.110044,0.067026,0.093206,0.081696,0.080620,0.089284,0.052139,0.064095,0.047635,0.058475,0.069645,0.063037,0.130803,0.054934,0.063283,0.059820,0.555304,0.590108,0.282219,0.402856,0.274748,0.312566,0.402806,0.422858,0.827731,0.539638,0.349590,0.313171,22.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2924.wav,66.384428,91.048773,595.946779,76.401230,1262.618358,1885.538377,3.336339,0.295737,27246.027638,101.792402,9.481316,59878.054615,56.123302,8.345351,8.992748e+05,442.160977,30.960859,0.029751,0.121434,-89.1994

In [358]:
totalData = pd.read_csv('.././data/development.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
totalData['gender'] = totalData['gender'].map(lambda x: 0 if x=='male' else 1)
totalData['tempo'] = totalData['tempo'].map(lambda x: float(x[1:-1]))
evaluate = pd.read_csv('.././data/evaluation.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
evaluate['path'] = evaluate['path'].map(lambda x: x.split('/')[-1])
evaluate['gender'] = evaluate['gender'].map(lambda x: 0 if x=='male'else 1)
evaluate['tempo'] = evaluate['tempo'].map(lambda x: float(x[1:-1]))

In [332]:
pd.Series(RandomForestRegressor(n_jobs=-1).fit(totalData.drop(columns=['age', 'ethnicity', 'path']), totalData['age'])
          .predict(evaluate.drop(columns=['ethnicity', 'path'])),
            index=evaluate.index, name='Predicted').to_csv('prediction.csv')

In [339]:
predictedAfrican = pd.Series(RandomForestRegressor(n_jobs=-1)
                    .fit(totalData[totalData['num_characters'] < 69].drop(columns=['path', 'ethnicity', 'age']), 
                         totalData[totalData['num_characters'] < 69]['age'])
                    .predict(evaluate[evaluate['num_characters'] < 69].drop(columns=['path', 'ethnicity'])),
                    index=evaluate[evaluate['num_characters'] < 69].index, name='Predicted')

In [341]:
predicted = pd.read_csv('prediction.csv', header=0, index_col=0)

In [344]:
for index in predicted.index:
    if index in predictedAfrican.index:
        predicted.loc[index] = predictedAfrican.loc[index]

In [ ]:
toVal = {item:parselmouth.Sound('.././data/audios_evaluation/'+item) for item in set(evaluate['path']).difference(set(predictedAfrican.index))}

In [360]:
dataVal = pd.concat([pd.DataFrame({file: pd.concat([
                                computeF0(toVal[file]), 
                                computeSpectralCharacteristic(toVal[file]),
                                # signalCharacteristics(toVal[file]), 
                                # computeRMSDuration(toVal[file]),
                                computeSilenceDuration(toVal[file]),
                                getMFCC(toVal[file]), 
                                computeJitterShimmerHNR(file, evaluate),
                                computeZCR(toVal[file]), 
                                compute_chroma(toVal[file].values[0], 22050)]
                                  ) for file in toVal}).T,
                        files.set_index(files['path'])['age']], axis=1)

C:\Users\utente\AppData\Local\Temp\ipykernel_42840\3502006958.py:22: RuntimeWarning: invalid value encountered in divide
  chroma = chroma / np.sum(chroma, axis=0, keepdims=True)  # Normalizzazione
C:\Users\utente\AppData\Local\Temp\ipykernel_42840\3502006958.py:22: RuntimeWarning: invalid value encountered in divide
  chroma = chroma / np.sum(chroma, axis=0, keepdims=True)  # Normalizzazione
C:\Users\utente\AppData\Local\Temp\ipykernel_42840\3502006958.py:22: RuntimeWarning: invalid value encountered in divide
  chroma = chroma / np.sum(chroma, axis=0, keepdims=True)  # Normalizzazione
C:\Users\utente\AppData\Local\Temp\ipykernel_42840\3502006958.py:22: RuntimeWarning: invalid value encountered in divide
  chroma = chroma / np.sum(chroma, axis=0, keepdims=True)  # Normalizzazione
C:\Users\utente\AppData\Local\Temp\ipykernel_42840\3502006958.py:22: RuntimeWarning: invalid value encountered in divide
  chroma = chroma / np.sum(chroma, axis=0, keepdims=True)  # Normalizzazione
C:\Users\u

KeyboardInterrupt: 

In [ ]:
dataVal2 = pd.concat([dataVal.drop(columns=['chroma_std_0', 'chroma_std_1', 'chroma_std_2',
       'chroma_std_3', 'chroma_std_4', 'chroma_std_5', 'chroma_std_6',
       'chroma_std_7', 'chroma_std_8', 'chroma_std_9', 'chroma_std_10',
       'chroma_std_11',
       'chroma_max_0',  'chroma_max_1', 'chroma_max_2',
       'chroma_max_3', 'chroma_max_4', 'chroma_std_5', 'chroma_max_6',
       'chroma_max_7', 'chroma_max_8', 'chroma_std_9', 'chroma_max_10',
       'chroma_max_11',
       'mfcc_max_0', 'mfcc_max_1', 'mfcc_max_2']), 
                      pd.Series(dataVal[['chroma_std_0', 'chroma_std_1', 'chroma_std_2',
       'chroma_std_3', 'chroma_std_4', 'chroma_std_5', 'chroma_std_6',
       'chroma_std_7', 'chroma_std_8', 'chroma_std_9', 'chroma_std_10',
       'chroma_std_11']].mean(axis=1), name='chromaStd'),
                      pd.Series(dataVal[['chroma_max_0',  'chroma_max_1', 'chroma_max_2',
       'chroma_max_3', 'chroma_max_4', 'chroma_std_5', 'chroma_max_6',
       'chroma_max_7', 'chroma_max_8', 'chroma_std_9', 'chroma_max_10',
       'chroma_max_11']].mean(axis=1), name='chromaMax'),
                      pd.Series(dataVal[['mfcc_max_0', 'mfcc_max_1', 'mfcc_max_2']].mean(axis=1), name='mfccMax')], axis=1)
dataVal2